In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"  # modern replacement for GPT-3.5
)
llm.invoke("Hi").content

c:\Users\Admin\Documents\ML_Projects\Agentic_AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Hello! How can I assist you today?'

In [31]:
message=[{"role":"system", "content":"you are an helpful assistant"},
          {"role":"user", "content":"Hi how are you?"}]

result = llm.invoke(message)
print(result.content)

I'm just a computer program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?


In [32]:
import re
import httpx

In [33]:
from typing import Any

class ChatBot:
    def __init__(self, system=""):
        self.system = system
        self.message = []
        if self.system:
            self.message.append({"role":"system", "content":system})

    def __call__(self, message):
        self.message.append({"role":"user","content":message})
        result = self.execute()
        self.message.append({"role":"assistant","content":result})
        return result

    def execute(self):
        llm = ChatOpenAI(model="gpt-4o-mini")
        result = llm.invoke(self.message)
        return result.content

In [34]:
assistant=ChatBot(system="you are a helpful assistant")

In [35]:
print(assistant("hi how are you?"))

Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


In [36]:
assistant("can you tell me how i can achieve my fitness goal?")

"Of course! Achieving a fitness goal involves a combination of planning, consistency, and motivation. Here are some steps to help you reach your fitness goal:\n\n1. **Define Your Goal**: Clearly specify what you want to achieve. Is it weight loss, muscle gain, improved endurance, flexibility, or overall health? Make your goal SMART (Specific, Measurable, Achievable, Relevant, Time-bound).\n\n2. **Create a Plan**: Develop a workout plan that aligns with your goal. This can include:\n   - **Strength training** for muscle gain or toning\n   - **Cardiovascular exercises** for weight loss or endurance\n   - **Flexibility and mobility work** such as yoga or stretching\n\n3. **Nutrition**: Evaluate your diet to support your fitness goals. Consider:\n   - Eating a balanced diet with the right macronutrients (proteins, carbohydrates, fats)\n   - Tracking your calorie intake if weight loss or gain is a goal\n   - Staying hydrated\n\n4. **Consistency**: Stick to your workout and nutrition plan. S

In [37]:
assistant.message

[{'role': 'system', 'content': 'you are a helpful assistant'},
 {'role': 'user', 'content': 'hi how are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"},
 {'role': 'user',
  'content': 'can you tell me how i can achieve my fitness goal?'},
 {'role': 'assistant',
  'content': "Of course! Achieving a fitness goal involves a combination of planning, consistency, and motivation. Here are some steps to help you reach your fitness goal:\n\n1. **Define Your Goal**: Clearly specify what you want to achieve. Is it weight loss, muscle gain, improved endurance, flexibility, or overall health? Make your goal SMART (Specific, Measurable, Achievable, Relevant, Time-bound).\n\n2. **Create a Plan**: Develop a workout plan that aligns with your goal. This can include:\n   - **Strength training** for muscle gain or toning\n   - **Cardiovascular exercises** for weight loss or endurance\n   -

In [38]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

In [39]:
prompt = """You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop, you output an Answer related to fitness, health, or exercise.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 1800 * 0.2
Runs a calculation (e.g., calorie intake, BMI, workout progress) and returns the result.

fitness_wiki:
e.g. fitness_wiki: Benefits of Cardio
Returns a summary of fitness-related information from a fitness knowledge base (Wikipedia or similar).

exercise_plan:
e.g. exercise_plan: Beginner full-body workout
Returns a recommended workout plan based on the type of exercise or user level.

nutrition_info:
e.g. nutrition_info: Protein sources for vegetarians
Looks up nutritional information or suggestions for diets.

Example session:
Question: What are the benefits of cardio exercises?
Thought: I should look up information about the benefits of cardio exercises.
Action: fitness_wiki: Benefits of Cardio
PAUSE

You will be called again with this:

Observation: Cardio exercises improve cardiovascular health, increase lung capacity, and aid in weight loss.

You then output:

Answer: Cardio exercises improve cardiovascular health, increase lung capacity, and help with weight loss.
""".strip()

In [40]:
action_re = re.compile('^Action: (\w+): (.*)')

In [41]:
import re

# Compiling the regular expression pattern
action_re = re.compile('^Action: (\w+): (.*)')

# Sample strings
text1 = "Action: Move: North"
text2 = "Action: Jump: High"
text3 = "Error: Not an Action"

In [42]:
# Testing the pattern
match1 = action_re.match(text1)

In [43]:
match1.group(1)

'Move'

In [44]:
match1.group(2)

'North'

In [45]:
match2 = action_re.match(text2)

In [46]:
match2

<re.Match object; span=(0, 18), match='Action: Jump: High'>

In [47]:
match3 = action_re.match(text3)

In [48]:
match3

In [49]:
# Extracting the matched groups
if match1:
    print(f"Matched: {match1.group(1)}, {match1.group(2)}")  # Move, North

Matched: Move, North


In [50]:
def wikipedia(q):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json"
    })
    return response.json()["query"]["search"][0]["snippet"]

In [51]:
def simon_blog_search(q):
    response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": q,
    })
    return response.json()[0]["text"]

In [52]:
def calculate(numbers):
    return eval(numbers)

In [53]:
calculate("10*7")

70

In [54]:
known_actions = {
    "wikipedia": wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search
}

In [55]:
question="hi how are you?"

In [56]:
bot = ChatBot(prompt)

In [57]:
bot

In [58]:
question="hello again?"

In [59]:
next_prompt = question

In [60]:
result = bot(next_prompt)
print(result)

Thought: It seems like I'm being prompted, but there's no specific question provided. I should wait for a question related to fitness, health, or exercise before proceeding. 

PAUSE


In [61]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]

In [62]:
actions

[]

In [63]:
def query(question, max_turns=5):
    i = 0
    bot = ChatBot(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return result

In [64]:
query("hello, what you doing?")

I'm here to help answer your questions related to fitness, health, or exercise! What would you like to know?


"I'm here to help answer your questions related to fitness, health, or exercise! What would you like to know?"

In [65]:
query("Has Simon written about AI?")

Thought: The question is about whether Simon has written about AI, which is not directly related to fitness, health, or exercise. I should focus on responding pertaining to fitness or exercise-related queries. 

Action: fitness_wiki: Benefits of strength training

PAUSE


Exception: Unknown action: fitness_wiki: Benefits of strength training

In [66]:
print(query("Fifteen * twenty five"))

Thought: I need to calculate the product of fifteen and twenty-five to provide the answer.  
Action: calculate: 15 * 25  
PAUSE  
 -- running calculate 15 * 25  
Observation: 375
Answer: The product of fifteen and twenty-five is 375.
Answer: The product of fifteen and twenty-five is 375.
